In [1]:
#import relevant libraries
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap

import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout

#NOTE: SUPPRESSES WARNINGS!
import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 14.95it/s]


Numba compilation complete!


In [28]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp

filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\2. Processed\\" 
savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\Appendixstats\\Climbing\\" 

openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"

In [29]:
if responder == "eOPN3" or "eOPN3 [ATR]":
    lst = ["46416jus", "vGAT", "nSyb", "OK371", "vGlut", "elav"]
    
if responder == "eOPN3-TS-ER":
    lst = ['elav', '46416jus', 'vGAT', 'nSyb']

if responder == "ACR [ATR]":
    lst = ["elav", "46416jus", "vGlut", "OK371"]
    
if responder == "ACR":
    lst = ['OK371']
    
if responder == "PdCO":
    lst = ['elav', 'OK371', 'nSyb']


print(lst)

['OK371']


## Thesis excel file generation

In [30]:
from dabest._stats_tools.confint_1group import summary_ci_1group

# Mapping from file responder name to display responder name
responder_display_map = {
    "eOPN3": "AsOPN3",
    "eOPN3 [ATR]": "AsOPN3 [ATR]",
    "eOPN3-TS-ER": "AsOPN3-TS-ER",
    "ACR": "GtACR1",
    "ACR [ATR]": "GtACR1 [ATR]",
    "PdCO": "PdCO",
}

def get_responder_display(responder):
    return responder_display_map.get(responder, responder)

def thesis_hedgesg_delta2(df, metric, df_naming, driver, responder):
    """
    Process delta2 data using hedges_g.
    For: df_sp, df_h
    Returns 4 rows: Control; Light off, Control; Light on, Test; Light off, Test; Light on
    """
    try:
        display_responder = get_responder_display(responder)
        
        # Filter out Recovery data and null values
        df6 = df[(df['ExperimentState'] != "Recovery")]
        name = []
        if any(df6[metric].isnull()):
            name = df6[df6[metric].isnull()]['index'].tolist()
        df_clean = df6[~df6['index'].isin(name)]
        
        # Load dabest for delta2 hedges_g
        db = dabest.load(
            data=df_clean, 
            x=["ExperimentState", "Type"], 
            y=metric,  
            delta2=True, 
            experiment="Type",
            experiment_label=['WT', 'Expt'], 
            x1_level=["Dark", "Full"], 
            paired="baseline", 
            id_col="index"
        )
        
        results = db.hedges_g.results
        delta_delta_results = db.hedges_g.delta_delta.results
        
        # Check if results are valid
        if len(results) == 0:
            return pd.DataFrame()
        
        # Determine which index is WT vs Expt
        if 'WT' in str(results.control.iloc[0]):
            wt_idx = 0
            expt_idx = 1
        else:
            wt_idx = 1
            expt_idx = 0
        
        # Get plot data for mean calculations
        plot_data = db._plot_data
        xvar = db._xvar
        yvar = db._yvar
        
        # Define the 4 groups
        groups = ["Dark WT", "Full WT", "Dark Expt", "Full Expt"]
        group_labels = ["Control; Light off", "Control; Light on", "Test; Light off", "Test; Light on"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        genotypes = [genotype_control, genotype_control, genotype_test, genotype_test]
        
        rows = []
        for i, (group, label, genotype) in enumerate(zip(groups, group_labels, genotypes)):
            # Get group data for mean calculation
            group_data = plot_data[plot_data[xvar] == group][yvar].values
            
            if len(group_data) == 0:
                continue
            
            # Calculate mean and CI
            group_stats = summary_ci_1group(
                x=group_data,
                func=np.mean,
                resamples=5000,
                alpha=0.05
            )
            
            mean_val = round(group_stats['summary'], 2)
            mean_ci_low = round(group_stats['bca_ci_low'], 2)
            mean_ci_high = round(group_stats['bca_ci_high'], 2)
            sample_size = len(group_data)
            
            # Effect Size - only for "Light on" rows (Full)
            if "Full" in group:
                if "WT" in group:
                    es_val = round(results.difference.iloc[wt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[wt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[wt_idx], 2)
                else:
                    es_val = round(results.difference.iloc[expt_idx], 2)
                    es_ci_low = round(results.bca_low.iloc[expt_idx], 2)
                    es_ci_high = round(results.bca_high.iloc[expt_idx], 2)
                delta_object = "Delta-g"
            else:
                es_val = " "
                es_ci_low = " "
                es_ci_high = " "
                delta_object = " "
            
            # Delta-g - only for Test; Light on
            if group == "Full Expt":
                dd_val = round(delta_delta_results.difference.iloc[0], 2)
                dd_ci_low = round(delta_delta_results.bca_low.iloc[0], 2)
                dd_ci_high = round(delta_delta_results.bca_high.iloc[0], 2)
            else:
                dd_val = " "
                dd_ci_low = " "
                dd_ci_high = " "
            
            rows.append({
                "Driver": driver,
                "Responder": display_responder,
                "Group": label,
                "Genotype": genotype,
                "Sample Size": sample_size,
                "Mean": mean_val,
                "Mean_CI_low": mean_ci_low,
                "Mean_CI_high": mean_ci_high,
                "Effect Size": es_val,
                "Effect Size_CI_low": es_ci_low,
                "Effect Size_CI_high": es_ci_high,
                "Delta Object": delta_object,
                "Delta-Delta/Delta-g": dd_val,
                "Delta-Delta/Delta-g_CI_low": dd_ci_low,
                "Delta-Delta/Delta-g_CI_high": dd_ci_high,
                "Metric": df_naming
            })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_hedgesg_delta2 for {driver} - {df_naming}: {e}")
        return pd.DataFrame()

In [31]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)

    # Thesis DataFrame creation
    thesis_dfs = [

        thesis_hedgesg_delta2(df_sp, "Velocity", "speed", driver, responder),
        thesis_hedgesg_delta2(df_h, "Y", "height", driver, responder),
    ]
    
    # Filter out empty DataFrames before concatenating
    thesis_dfs = [df for df in thesis_dfs if len(df) > 0]
    
    if len(thesis_dfs) > 0:
        df_thesis_final = pd.concat(thesis_dfs, ignore_index=True)
        df_thesis_final.to_csv(savedir + n + " x " + responder + "_thesis_stats.csv", index=False)
    else:
        print(f"  Warning: No valid data for {driver}")
    
print("Done!")

OK371
Done!
